# Deploy Kimi-K2.7-Code on an Amazon SageMaker AI Inference Component (ml.g7e.48xlarge) with vLLM

This example deploys [moonshotai/Kimi-K2.7-Code](https://huggingface.co/moonshotai/Kimi-K2.7-Code) on a single `ml.g7e.48xlarge` (8x NVIDIA RTX PRO 6000 Blackwell, 8x96 GB) using **SageMaker Inference Components** and the **SageMaker vLLM Deep Learning Container**.


## Configuration

In [ ]:
instance = {"type": "ml.g7e.48xlarge", "num_gpu": 8}
model_id = "moonshotai/Kimi-K2.7-Code"

model_name = f"kimi-k27-code-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
ic_name = f"{model_name}-ic"
variant_name = "v1"

# ~595 GB of weights download from the HF Hub inside the container start-up health-check window — use the API maximum (3600 s).
startup_timeout = 3600


In [1]:
%pip install --upgrade --quiet --no-warn-conflicts boto3

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import re
import sys
import json
import time
import codecs
import boto3
from botocore.config import Config
from botocore.exceptions import EventStreamError
from IPython.display import display, Markdown, clear_output

boto_session = boto3.Session()
region = boto_session.region_name
assert region, "No AWS Region configured: set AWS_DEFAULT_REGION (or a profile region) before continuing"

sm = boto3.client("sagemaker")  # control plane: models, endpoints, inference components

# Data plane. A thinking model can generate for minutes, so raise boto3's default 60 s read timeout
# and disable automatic retries — a retried generation would silently double the latency and occupy GPU capacity for a second full generation.
sm_runtime = boto3.client(
    "sagemaker-runtime",
    config=Config(read_timeout=900, connect_timeout=60, retries={"max_attempts": 0}),
)
print(f"region: {region}")

region: us-east-2


In [ ]:
#
# Helper functions to remove dependency on SageMaker Python SDK
#
def get_sagemaker_role():
    """Resolve the execution role ARN of the current identity.

    The STS assumed-role ARN drops the role's IAM path (e.g. `/service-role/`), so rebuilding the role ARN
    from it yields an ARN SageMaker rejects when the role has a path. Look the role up in IAM first and fall
    back to the STS-derived form only when the caller lacks iam:GetRole.
    """
    arn = boto3.client("sts").get_caller_identity()["Arn"]
    match = re.match(r"^arn:(aws[^:]*):sts::(\d+):assumed-role/([^/]+)/", arn)
    if not match:
        raise ValueError(f"Not running under an assumed role; set `role` explicitly (caller: {arn})")
    partition, account, role_name = match.groups()
    try:
        return boto3.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]
    except Exception:
        return f"arn:{partition}:iam::{account}:role/{role_name}"


def _wait_for_resource(describe_fn, name_key, status_key, label, name, sleep_time=60):
    """Poll a SageMaker resource until it leaves the 'Creating' or 'Updating' state; raise if it ends in a failed state."""
    progress = ""
    while True:
        desc = describe_fn(**{name_key: name})
        status = desc[status_key]
        if status not in ("Creating", "Updating"):
            break
        progress += "."
        clear_output(wait=True)
        print(f"Waiting for '{name}': {progress}")
        time.sleep(sleep_time)
    print(f"{label}: '{name}', Status: '{status}'")
    if status.endswith("Failed"):  # 'Failed' (endpoint or IC) and 'UpdateRollbackFailed' (endpoint)
        reason = desc.get("FailureReason", "no FailureReason returned")
        print(f"FailureReason: {reason}")
        raise RuntimeError(f"{label} '{name}' ended in status '{status}': {reason}")


def wait_for_endpoint(endpoint_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_endpoint, "EndpointName", "EndpointStatus",
        "Endpoint", endpoint_name, sleep_time,
    )


def wait_for_ic(ic_name: str, sleep_time: int = 60):
    _wait_for_resource(
        sm.describe_inference_component, "InferenceComponentName", "InferenceComponentStatus",
        "IC", ic_name, sleep_time,
    )

In [ ]:
#
# Overwrite with your role ARN
#
role = None

if role is None:
    role = get_sagemaker_role()
print(role)

One `ml.g7e.48xlarge` hosts the model at tensor parallelism 8.

In [ ]:
instance = {"type": "ml.g7e.48xlarge", "num_gpu": 8}
model_id = "moonshotai/Kimi-K2.7-Code"

model_name = f"kimi-k27-code-{time.strftime('%y%m%d-%H%M%S')}"
endpoint_name = model_name
endpoint_config_name = model_name
ic_name = f"{model_name}-ic"
variant_name = "v1"

# ~595 GB of weights download from the HF Hub inside the container start-up health-check window — use the API maximum (3600 s).
startup_timeout = 3600


## Container and vLLM configuration

The SageMaker [vLLM DLC](https://aws.github.io/deep-learning-containers/vllm/) turns every `SM_VLLM_*` environment variable into a `vllm serve` flag: `SM_VLLM_MAX_MODEL_LEN=131072` becomes `--max-model-len 131072`, a value of `true` becomes a bare boolean flag (`SM_VLLM_TRUST_REMOTE_CODE=true` → `--trust-remote-code`), `false` omits the flag, a JSON object is passed through as a single argument (e.g. `SM_VLLM_LIMIT_MM_PER_PROMPT='{"image": 4}'`), and a JSON array becomes one argument per element. The container listens on port 8080 and exposes the OpenAI chat-completions schema on `/invocations`.

The flags below follow the vLLM recipe for this model (`--tensor-parallel-size 8 --mm-encoder-tp-mode data --trust-remote-code --tool-call-parser kimi_k2 --enable-auto-tool-choice --reasoning-parser kimi_k2`) — minus `--mm-encoder-tp-mode data`, which only matters when the vision encoder is loaded (see `SM_VLLM_LANGUAGE_MODEL_ONLY`) — plus the sizing choices for this instance:

| Setting | Why |
| :--- | :--- |
| `SM_VLLM_MAX_MODEL_LEN=131072` | Half the native 262,144. See the KV-cache budget below — a single 262,144-token sequence needs ≈ 17.2 GiB of KV cache, essentially all of the memory left after the weights. |
| `SM_VLLM_GPU_MEMORY_UTILIZATION=0.92` | Fraction of each GPU's memory (~95.6 GiB as reported by CUDA) vLLM may claim for weights + activations + KV cache. The weights (~71 GiB), activations and CUDA graphs are fixed costs, so this fraction only resizes what is left for KV cache. If start-up fails with vLLM's `... is larger than the available KV cache memory` or `No available memory for the cache blocks` error, the budget cannot hold the fixed costs plus one 131,072-token request: lower `SM_VLLM_MAX_MODEL_LEN` (e.g. 65536) or raise this value toward 0.95 — lowering it shrinks only the KV budget and makes that error worse. Lower it only for `Free memory on device ... is less than desired GPU memory utilization` or a hard CUDA out-of-memory raised outside vLLM's budget (NCCL buffers, CUDA-graph capture). |
| `SM_VLLM_MAX_NUM_SEQS=32` | Scheduler cap on requests decoded in the same step, lowered from vLLM's default of 1024 on this GPU class (>= 70 GiB). It bounds the sampler/logits buffers profiled at start-up (~640 MiB for the fp32 logits buffer at 1024 requests × the 163,840-token vocabulary, ~20 MiB at 32; the sampler warm-up allocates a few such buffers) and the CUDA-graph batch sizes captured — faster start-up and a little less reserved memory; peak activation memory is governed by `--max-num-batched-tokens` (default 8192, left unchanged). It does not create KV capacity: the KV cache below is what bounds effective concurrency (181,920 tokens ≈ 5–22 requests at 8–32K each; automatic prefix caching across agent turns stretches this). When KV blocks run out vLLM preempts or queues requests rather than failing; the start-up line `Maximum concurrency for 131,072 tokens per request: N.NNx` (printed by the log helper below) gives the measured figure. |
| `SM_VLLM_REASONING_PARSER=kimi_k2` | Thinking is always on (the prompt already opens `<think>`); the parser splits the thinking off into the `reasoning` field instead of leaving the raw thinking text and its closing `</think>` in `content`. |
| `SM_VLLM_TOOL_CALL_PARSER=kimi_k2` + `SM_VLLM_ENABLE_AUTO_TOOL_CHOICE=true` | OpenAI-style `tools` / `tool_calls` support. |
| `SM_VLLM_LANGUAGE_MODEL_ONLY=true` | Serve text only: the ~400M-parameter MoonViT encoder and projector are not built or loaded at all and the multimodal profiling pass is skipped, leaving the most memory for KV cache. Remove it (and add the two commented multimodal settings) to accept images. |

**KV-cache budget (per GPU, approximate, in GiB — the unit vLLM logs in).** vLLM measures these exactly at start-up and logs `GPU KV cache size: N tokens` and `Maximum concurrency for 131,072 tokens per request`.

| Item | Estimate |
| :--- | :--- |
| GPU memory | 96 GB marketed; vLLM reports 94.97 GiB per GPU, which is what `gpu-memory-utilization` is applied to |
| vLLM budget at 0.92 | 87.37 GiB (logged as `Desired GPU memory utilization is (0.92, 87.37 GiB)`) |
| Weights at TP=8 | 595 GB / 8 ≈ 74.4 GB ≈ 69.3 GiB, plus ~2.2 GB (~2 GiB) of unquantized tensors that vLLM replicates on every rank (`q_a_proj`, `kv_a_proj_with_mqa` and the MoE router gate) and NCCL/CUDA buffers — vLLM measured **73.28 GiB** "consumed memory (weights + non-torch)" |
| Activations and CUDA graphs | 2.19 GiB peak activations (profiled at `max_num_batched_tokens=8192`) + 0.55 GiB CUDA-graph pool, measured |
| KV cache per token (MLA, 16-bit cache) | (512 latent + 64 rope) × 2 bytes × 61 layers = 70,272 B ≈ 68.6 KiB — the MLA latent is replicated on every TP rank, not sharded |
| KV capacity | **11.91 GiB = 181,920 tokens** (`GPU KV cache size: 181,920 tokens, Maximum concurrency for 131,072 tokens per request: 1.39x`). A single 262,144-token request would need ≈ 17.2 GiB, which is why `max_model_len=131072`. In practice: one 131K request plus a few short ones, or roughly 5–22 concurrent requests at 8–32K tokens each (automatic prefix caching across agent turns stretches this). |

In [4]:
inference_image = f"763104351884.dkr.ecr.{region}.amazonaws.com/vllm:0.28.0-gpu-py312-cu130-ubuntu24.04-sagemaker"

common_env = {}
# Optional: the checkpoint is public, but a Hugging Face token raises Hub rate limits for the 64-shard download.
# The token is stored in plaintext in the SageMaker Model definition (visible via DescribeModel): use a read-only
# fine-grained token, or leave it unset.
if os.environ.get("HF_TOKEN"):
    common_env["HF_TOKEN"] = os.environ["HF_TOKEN"]

vllm_env = {
    "SM_VLLM_MODEL": model_id,
    "SM_VLLM_TENSOR_PARALLEL_SIZE": json.dumps(instance["num_gpu"]),
    "SM_VLLM_MAX_MODEL_LEN": "131072",         # model maximum is 262144 — see the KV-cache budget above
    "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.92",
    "SM_VLLM_MAX_NUM_SEQS": "32",             # scheduler cap; the KV cache is the real concurrency limit — see budget above
    "SM_VLLM_TRUST_REMOTE_CODE": "true",       # the tokenizer (tokenization_kimi.py TikTokenTokenizer) is custom code; vLLM has a native KimiK25 config
    "SM_VLLM_REASONING_PARSER": "kimi_k2",     # thinking is always on: returned as `reasoning`
    "SM_VLLM_TOOL_CALL_PARSER": "kimi_k2",
    "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
    "SM_VLLM_LANGUAGE_MODEL_ONLY": "true",     # text-only serving: MoonViT is not loaded; maximizes KV cache
    # To accept images, delete SM_VLLM_LANGUAGE_MODEL_ONLY and add:
    # "SM_VLLM_MM_ENCODER_TP_MODE": "data",              # run the small MoonViT encoder data-parallel, not TP-sharded
    # "SM_VLLM_LIMIT_MM_PER_PROMPT": json.dumps({"image": 4}),
}
env = common_env | vllm_env
print(json.dumps(vllm_env, indent=2))

{
  "SM_VLLM_MODEL": "moonshotai/Kimi-K2.7-Code",
  "SM_VLLM_TENSOR_PARALLEL_SIZE": "8",
  "SM_VLLM_MAX_MODEL_LEN": "131072",
  "SM_VLLM_GPU_MEMORY_UTILIZATION": "0.92",
  "SM_VLLM_MAX_NUM_SEQS": "32",
  "SM_VLLM_TRUST_REMOTE_CODE": "true",
  "SM_VLLM_REASONING_PARSER": "kimi_k2",
  "SM_VLLM_TOOL_CALL_PARSER": "kimi_k2",
  "SM_VLLM_ENABLE_AUTO_TOOL_CHOICE": "true",
  "SM_VLLM_LANGUAGE_MODEL_ONLY": "true"
}


## Deployment

Three steps, in order:
1. **Endpoint config + endpoint**
2. **Model** — the vLLM container plus environment.
3. **Inference component** — attaches the model to the endpoint with an explicit claim on all 8 accelerators.

In [ ]:
_ = sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ExecutionRoleArn=role,
    ProductionVariants=[
        {
            "VariantName": variant_name,
            "InstanceType": instance["type"],
            "InitialInstanceCount": 1,
            "InferenceAmiVersion": "al2023-ami-sagemaker-inference-gpu-4-1",  # driver 580 / CUDA 13.0, per the DLC docs; remove if rejected for this instance type
            "VariantInstanceProvisionTimeoutInSeconds": 3600,  # keep retrying for g7e.48xlarge capacity for up to an hour
            "ManagedInstanceScaling": {"Status": "ENABLED", "MinInstanceCount": 1, "MaxInstanceCount": 1},
            "RoutingConfig": {"RoutingStrategy": "LEAST_OUTSTANDING_REQUESTS"},
        },
    ],
)

_ = sm.create_endpoint(EndpointName=endpoint_name,
                       EndpointConfigName=endpoint_config_name)

wait_for_endpoint(endpoint_name)

Waiting for 'kimi-k27-code-260910-032255': ...
Endpoint: 'kimi-k27-code-260910-032255', Status: 'InService'


In [ ]:
_ = sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=role,
    PrimaryContainer={
        "Image": inference_image,
        "Environment": env,
    },
)

In [6]:
_ = sm.create_inference_component(
    InferenceComponentName=ic_name,
    EndpointName=endpoint_name,
    VariantName=variant_name,
    Specification={
        "ModelName": model_name,
        "StartupParameters": {
            "ContainerStartupHealthCheckTimeoutInSeconds": startup_timeout,
        },
        "ComputeResourceRequirements": {
            "NumberOfAcceleratorDevicesRequired": instance["num_gpu"],  # all 8 GPUs — no GPU is left for another GPU-backed component
            # ml.g7e.48xlarge: 192 vCPU, 2048 GiB RAM — this claim (100 cores, ~931 GiB) leaves 92 vCPU / ~1.1 TiB for CPU-only components
            "NumberOfCpuCoresRequired": 100,
            "MinMemoryRequiredInMb": 1000000,
        },
    },
    RuntimeConfig={"CopyCount": 1},
)

wait_for_ic(ic_name)

Waiting for 'kimi-k27-code-260910-032255-ic': ...................
IC: 'kimi-k27-code-260910-032255-ic', Status: 'InService'


In [ ]:
def print_startup_logs(pattern=r"KV cache|Maximum concurrency|vLLM server arguments|Loading weights took|ERROR|Error|OOM|out of memory", limit=40):
    """Print matching vLLM start-up log lines for the inference component (needs CloudWatch Logs read access)."""
    logs = boto3.client("logs")
    group = f"/aws/sagemaker/InferenceComponents/{ic_name}"
    try:
        streams = logs.describe_log_streams(logGroupName=group, orderBy="LastEventTime", descending=True, limit=5)["logStreams"]
        shown = 0
        for stream in streams:
            # read the stream from the beginning and follow the pagination token: an 8-rank start-up with
            # download progress logging can exceed the 10,000-event / 1 MB window of a single call
            kwargs = {"logGroupName": group, "logStreamName": stream["logStreamName"], "startFromHead": True, "limit": 10000}
            for _ in range(200):
                page = logs.get_log_events(**kwargs)
                for e in page["events"]:
                    if re.search(pattern, e["message"]):
                        print(e["message"].rstrip()[:400])
                        shown += 1
                        if shown >= limit:
                            return
                if not page["events"] or page["nextForwardToken"] == kwargs.get("nextToken"):
                    break
                kwargs["nextToken"] = page["nextForwardToken"]
        if shown == 0:
            print(f"no matching lines yet in {group}")
    except Exception as exc:  # missing permissions or the log group does not exist yet
        print(f"could not read {group}: {exc}")


print_startup_logs()

INFO: vLLM server arguments: ['--port', '8080', '--enable-auto-tool-choice', '--gpu-memory-utilization', '0.92', '--language-model-only', '--max-model-len', '131072', '--max-num-seqs', '32', '--model', 'moonshotai/Kimi-K2.7-Code', '--reasoning-parser', 'kimi_k2', '--tensor-parallel-size', '8', '--tool-call-parser', 'kimi_k2', '--trust-remote-code']
(Worker_TP0 pid=1875) INFO 09-10 03:42:26 [default_loader.py:430] Loading weights took 250.46 seconds
(Worker_TP4 pid=1879) INFO 09-10 03:43:33 [gpu_worker.py:593] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9200 is equivalent to --gpu-memory-utilization=0.9132 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9268. To disable, set VLLM_MEMORY_PROFILER_ESTIMA
(Worker_TP5 pid=1880) INFO 09-10 03:43:33 [gpu_worker.py:593] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-me

## Inference

Requests target the endpoint **plus** the inference component (`InferenceComponentName`). The container serves the OpenAI chat-completions schema on `/invocations`; with the `kimi_k2` reasoning parser the model's thinking comes back in the message's `reasoning` field and the answer in `content`.

Two helpers:
- `chat()` — plain request/response. SageMaker real-time invocations must complete within **60 seconds**, and a thinking model spends tokens before its first visible output — so keep these calls short (focused prompt, small `max_tokens`): on this PCIe-attached (no NVLink) TP=8 deployment single-stream decode measured ~56 tokens/s, so 1024 tokens take ~18 s of decoding (the 680-token answer below returned in 14.8 s).
- `chat_stream()` — server-sent-events streaming via `invoke_endpoint_with_response_stream`. Use this for long generations and for the agent loop; it prints tokens as they arrive and reassembles the final message (reasoning, content, tool calls, usage, TTFT and decode speed). Streaming lifts the timeout to **~8mins**. SageMaker can still end a stream (`ModelStreamError` with `ModelInvocationTimeExceeded` or `StreamBroken`), which boto3 raises as `botocore.exceptions.EventStreamError`.

In [ ]:
def invoke(payload):
    res = sm_runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        InferenceComponentName=ic_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )
    return json.loads(res["Body"].read().decode("utf-8"))


def chat(messages, max_tokens=1024, **kwargs):
    """Non-streaming chat completion with the model card's recommended sampling."""
    payload = {"messages": messages, "max_tokens": max_tokens, "temperature": 1.0, "top_p": 0.95, **kwargs}
    return invoke(payload)


def reasoning_of(message):
    return message.get("reasoning") or message.get("reasoning_content")


def show(response):
    choice = response["choices"][0]
    message = choice["message"]
    if reasoning_of(message):
        display(Markdown("### Reasoning\n---"))
        display(Markdown(reasoning_of(message)))
    if message.get("content"):
        display(Markdown("### Answer\n---"))
        display(Markdown(message["content"]))
    print(f"\nfinish_reason={choice['finish_reason']}  usage={response['usage']}")
    if choice["finish_reason"] == "length":
        print("hint: completion budget exhausted"
              + (" during reasoning — no answer was produced" if not message.get("content") else " — answer truncated")
              + "; use chat_stream() with a larger max_tokens")

### Non-streaming with reasoning

In [8]:
messages = [
    {
        "role": "user",
        "content": (
            "In Python, when should I prefer functools.lru_cache over a hand-written dict cache? "
            "Answer in under 120 words with one short code example."
        ),
    }
]

t0 = time.time()
try:
    response = chat(messages, max_tokens=1024)
except sm_runtime.exceptions.ModelError as exc:
    print(f"invocation failed after {time.time() - t0:.1f}s: {exc}\n"
          "If this is a timeout, lower max_tokens or use chat_stream() below.")
    raise
print(f"latency: {time.time() - t0:.1f}s")
show(response)

latency: 14.8s


### Reasoning
---

 We need answer Python user: when prefer functools.lru_cache over hand-written dict cache. Answer under 120 words with one short code example. Need concise. Consider quick. 

We should draft under 120 words. Include code example short.

Potential answer:
Use functools.lru_cache when you want a simple, bounded, thread-safe memoizer with least-recently-used eviction and hashable arguments. Prefer it over dict when eviction policy, max size, and recursive caching matter; use a dict only when keys/values are non-hashable, complex invalidation logic, or you need more control.

Code example:
@lru_cache(maxsize=None)
def fib(n):
    if n < 2: return n
    return fib(n-1)+fib(n-2)

Need count words. Let's draft under 120.

Word count check: Use functools.lru_cache when you need a simple, bounded memoizer with automatic LRU eviction, thread safety, and recursive caching for hashable arguments. Use a hand-written dict only when keys are non-hashable, you need custom invalidation, or fine-grained control over storage. Example:
from functools import lru_cache
@lru_cache(maxsize=128)
def fib(n):
    if n < 2: return n
    return fib(n - 1) + fib(n - 2)

Count: Use functools.lru_cache(4) when(1) you(2) need(3) a(4) simple,(5) bounded(6) memoizer(7) with(8) automatic(9) LRU(10) eviction,(11) thread(12) safety,(13) and(14) recursive(15) caching(16) for(17) hashable(18) arguments.(19) Use(20) a(21) hand-written(22) dict(23) only(24) when(25) keys(26) are(27) non-hashable,(28) you(29) need(30) custom(31) invalidation,(32) or(33) fine-grained(34) control(35) over(36) storage.(37) Example:(38) from(39) functools(40) import(41) lru_cache(42) @lru_cache(maxsize=128)(43?) def fib(n):(44?) if(45) n(46) <(47) 2:(48) return(49) n(50) return(51) fib(n(52) -(53) 1)(54) +(55) fib(n(56) -(57) 2)(58)
Actually code tokens count as words roughly. Under 120 fine.

Ensure answer fits. Need maybe mention Python versions. Keep short.

### Answer
---

 Prefer `functools.lru_cache` when you want a simple, bounded, thread-safe memoizer with automatic LRU eviction and recursive caching for hashable arguments. Use a hand-written `dict` only when keys are non-hashable, you need custom invalidation, or you want fine-grained control over storage.

```python
from functools import lru_cache

@lru_cache(maxsize=128)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)
```


finish_reason=stop  usage={'prompt_tokens': 38, 'total_tokens': 718, 'completion_tokens': 680, 'prompt_tokens_details': None, 'completion_tokens_details': {'reasoning_tokens': 565}}


### Streaming with reasoning

In [ ]:
def chat_stream(messages, max_tokens=4096, echo=True, **kwargs):
    """Stream a chat completion; return the reassembled assistant message plus usage and timing metrics."""
    payload = {
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 1.0,
        "top_p": 0.95,
        "stream": True,
        "stream_options": {"include_usage": True},
        **kwargs,
    }
    res = sm_runtime.invoke_endpoint_with_response_stream(
        EndpointName=endpoint_name,
        InferenceComponentName=ic_name,
        ContentType="application/json",
        Body=json.dumps(payload),
    )

    t0, t_first = time.time(), None
    out = {"role": "assistant", "reasoning": "", "content": "", "tool_calls": {}, "finish_reason": None, "usage": None}
    phase = None
    decoder = codecs.getincrementaldecoder("utf-8")()  # a PayloadPart can end in the middle of a multi-byte character
    buffer = ""

    def emit(kind, text):
        nonlocal phase
        if not echo:
            return
        if phase != kind:
            print(f"\n\n[{kind}]", flush=True)
            phase = kind
        print(text, end="", flush=True)

    try:
        for event in res["Body"]:
            if "PayloadPart" not in event:
                continue
            buffer += decoder.decode(event["PayloadPart"]["Bytes"])
            while "\n" in buffer:
                line, buffer = buffer.split("\n", 1)
                line = line.strip()
                if not line.startswith("data:"):
                    continue
                data = line[len("data:"):].strip()
                if data == "[DONE]":
                    continue
                chunk = json.loads(data)
                if "error" in chunk:  # vLLM reports errors that occur after the SSE stream has started as a data chunk
                    raise RuntimeError(f"vLLM stream error: {chunk['error']}")
                if chunk.get("usage"):
                    out["usage"] = chunk["usage"]
                for choice in chunk.get("choices", []):
                    if choice.get("finish_reason"):
                        out["finish_reason"] = choice["finish_reason"]
                    delta = choice.get("delta") or {}
                    reasoning = delta.get("reasoning") or delta.get("reasoning_content")
                    if reasoning:
                        t_first = t_first or time.time()
                        out["reasoning"] += reasoning
                        emit("reasoning", reasoning)
                    if delta.get("content"):
                        t_first = t_first or time.time()
                        out["content"] += delta["content"]
                        emit("answer", delta["content"])
                    for tc in delta.get("tool_calls") or []:
                        t_first = t_first or time.time()
                        slot = out["tool_calls"].setdefault(
                            tc.get("index", 0), {"id": None, "type": "function", "function": {"name": "", "arguments": ""}}
                        )
                        if tc.get("id"):
                            slot["id"] = tc["id"]
                        fn = tc.get("function") or {}
                        if fn.get("name"):
                            slot["function"]["name"] = fn["name"]
                        if fn.get("arguments"):
                            slot["function"]["arguments"] += fn["arguments"]
        buffer += decoder.decode(b"", final=True)  # raises if the stream ended inside a multi-byte character
    except EventStreamError as exc:
        # SageMaker ended the stream (ModelStreamError: ModelInvocationTimeExceeded / StreamBroken, or InternalStreamFailure).
        # botocore raises these as exceptions rather than delivering them as events. Report the partial result, then re-raise
        # so callers (e.g. the agent loop) do not treat a half-received message as complete.
        print(f"\n\nstream ended by SageMaker after {time.time() - t0:.1f}s: {exc}\n"
              f"received so far: {len(out['reasoning'])} reasoning chars, {len(out['content'])} answer chars, "
              f"{len(out['tool_calls'])} tool call(s)", flush=True)
        raise
    t_end = time.time()

    out["tool_calls"] = [out["tool_calls"][i] for i in sorted(out["tool_calls"])] or None
    if t_first is not None:
        out["metrics"] = {"ttft_s": round(t_first - t0, 2), "total_s": round(t_end - t0, 1)}
        if out["usage"]:
            out["metrics"]["decode_tok_s"] = round(out["usage"]["completion_tokens"] / max(t_end - t_first, 1e-6), 1)
    if echo:
        print(f"\n\nfinish_reason={out['finish_reason']}  usage={out['usage']}  metrics={out.get('metrics')}")
    return out

In [9]:
messages = [
    {
        "role": "user",
        "content": (
            "Explain the difference between asyncio.gather and asyncio.TaskGroup in Python 3.11+, "
            "then show a minimal example of each."
        ),
    }
]
result = chat_stream(messages, max_tokens=4096)



[reasoning]
 We need answer difference between asyncio.gather and asyncio.TaskGroup in Python 3.11+, minimal examples. Need explain. Let's think carefully. User wants explanation and minimal examples. Need ensure correctness.

asyncio.gather: schedule coroutines concurrently, returns aggregate list of results in original order. If return_exceptions=True, exceptions returned in result list; otherwise first exception propagates, other tasks continue (unless cancelled?). Actually with gather, if one task raises and return_exceptions=False, the exception is propagated to the caller immediately; however other tasks are not cancelled automatically, they continue running? Wait: gather's behavior: if one task raises and return_exceptions is False, the gather() future is marked with exception but other tasks continue. The caller must await gather again? No, once the future has exception, awaiting it raises exception. The remaining tasks are not cancelled. There is some nuance: in Python 3.10?

## Tool calling: a minimal coding agent

Kimi-K2.7-Code is built for long-horizon agentic coding: it thinks, calls tools, reads the results, and keeps going. This section gives it four tools that operate in a throw-away workspace directory — write/read/list files and run Python — and loops until it stops calling tools.

Two details specific to this model family:
- **Pass the reasoning back.** Thinking is preserved across turns (`preserve_thinking` is forced on). The chat template renders each assistant turn's `reasoning` into `<think>…</think>`, so every assistant message we append to the history carries its `reasoning` alongside `content` and `tool_calls`.
- **Tool-call IDs** look like `functions.write_file:0`; return each result as a `tool` message with the same `tool_call_id`.

> **Safety note.** `run_python` is **not sandboxed**: it runs model-generated code in a subprocess on *this* notebook host with only a 60 s timeout and `cwd=WORKSPACE`. The subprocess gets a minimal environment (no AWS credentials or session variables are passed through), but it keeps the host's filesystem and network access. Only the file tools (`write_file`/`read_file`/`list_files`) are confined to the workspace directory via path checks. Run this section only in a disposable environment you are willing to discard, and review the code the model writes. In this demo task finished in 3 turns in about 2 minutes (write both files, run the tests, summarize); budget 5–15 minutes for harder tasks, lower `max_turns`/`max_tokens` for a quicker smoke test, and interrupt the kernel to stop — the in-flight request finishes on the server but no further turns are sent. The workspace is removed in the cleanup cell.

In [10]:
import pathlib
import shlex
import shutil
import subprocess
import tempfile

WORKSPACE = pathlib.Path(tempfile.mkdtemp(prefix="kimi-agent-")).resolve()
print(f"workspace: {WORKSPACE}")


def _safe_path(relative_path: str) -> pathlib.Path:
    path = (WORKSPACE / relative_path).resolve()
    if path != WORKSPACE and WORKSPACE not in path.parents:
        raise ValueError(f"path escapes the workspace: {relative_path}")
    return path


def write_file(path: str, content: str) -> str:
    target = _safe_path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"wrote {len(content)} characters to {path}"


def read_file(path: str) -> str:
    return _safe_path(path).read_text()


def list_files() -> str:
    files = sorted(str(p.relative_to(WORKSPACE)) for p in WORKSPACE.rglob("*") if p.is_file() and "__pycache__" not in p.parts)
    return "\n".join(files) or "(workspace is empty)"


def run_python(command: str) -> str:
    """Run `python <command>` in the workspace, e.g. 'stats.py' or '-m unittest -v'."""
    args = shlex.split(command)
    if args and args[0] in ("python", "python3"):  # the model sometimes includes the interpreter itself
        args = args[1:]
    proc = subprocess.run(
        [sys.executable, *args],
        cwd=WORKSPACE,
        capture_output=True,
        text=True,
        timeout=60,
        # minimal environment: no AWS credentials or session variables leak into model-generated code
        env={"PATH": os.environ.get("PATH", ""), "HOME": str(WORKSPACE), "LANG": "C.UTF-8", "PYTHONDONTWRITEBYTECODE": "1"},
    )
    # truncate each stream separately, keeping the tail, so a verbose stdout cannot push the traceback / unittest report out of the result
    def tail(text, limit=3000):
        return text if len(text) <= limit else f"... [{len(text) - limit} characters truncated] ...\n{text[-limit:]}"

    return f"exit code: {proc.returncode}\n--- stdout ---\n{tail(proc.stdout)}\n--- stderr ---\n{tail(proc.stderr)}"


TOOL_FUNCTIONS = {"write_file": write_file, "read_file": read_file, "list_files": list_files, "run_python": run_python}

TOOLS = [
    {"type": "function", "function": {
        "name": "write_file",
        "description": "Create or overwrite a text file in the workspace.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "Relative path inside the workspace"},
            "content": {"type": "string", "description": "Full file content"}},
            "required": ["path", "content"]}}},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read a text file from the workspace.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "Relative path inside the workspace"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "list_files",
        "description": "List all files in the workspace.",
        "parameters": {"type": "object", "properties": {}}}},
    {"type": "function", "function": {
        "name": "run_python",
        "description": "Run the Python interpreter in the workspace with the given arguments, e.g. 'stats.py', '-m unittest -v' or '-c \"print(1)\"' (a leading 'python' is tolerated). Returns exit code, stdout and stderr.",
        "parameters": {"type": "object", "properties": {
            "command": {"type": "string", "description": "Arguments passed to the python interpreter"}},
            "required": ["command"]}}},
]

workspace: /tmp/kimi-agent-6lhpvbxg


In [ ]:
SYSTEM_PROMPT = (
    "You are a senior software engineer working in a scratch workspace directory. "
    "Use the tools to create files and run code; never just describe what you would do. "
    "Run the tests after writing them, fix any failures, and finish with a short summary of what you built."
)


def run_agent(task: str, max_turns: int = 12, max_tokens: int = 8192):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": task}]

    for turn in range(1, max_turns + 1):
        print(f"\n{'=' * 24} turn {turn} {'=' * 24}")
        try:
            reply = chat_stream(messages, max_tokens=max_tokens, echo=False, tools=TOOLS, tool_choice="auto")
        except EventStreamError as exc:
            # chat_stream() already printed the partial result; keep the transcript so far and stop cleanly
            print(f"\nstopped: stream ended by SageMaker on turn {turn}: {exc}")
            return None, messages

        if reply["reasoning"]:
            preview = reply["reasoning"].strip().replace("\n", " ")
            print(f"[reasoning] {preview[:300]}{'…' if len(preview) > 300 else ''}")
        if reply["content"]:
            print(f"[answer] {reply['content'].strip()[:1500]}")

        assistant = {"role": "assistant", "content": reply["content"] or None, "reasoning": reply["reasoning"]}
        if reply["tool_calls"]:
            assistant["tool_calls"] = reply["tool_calls"]
        messages.append(assistant)

        if not reply["tool_calls"]:
            if reply["finish_reason"] == "length":
                print(f"\nstopped: the turn was truncated at max_tokens={max_tokens} (finish_reason=length); last usage: {reply['usage']}")
            else:
                print(f"\nfinished after {turn} turn(s); last usage: {reply['usage']}")
            return reply["content"], messages

        for tc in reply["tool_calls"]:
            name = tc["function"]["name"]
            try:
                args = json.loads(tc["function"]["arguments"] or "{}")
                result = TOOL_FUNCTIONS[name](**args)
            except Exception as exc:
                args = tc["function"]["arguments"]
                result = f"ERROR: {type(exc).__name__}: {exc}"
            print(f"[tool] {name}({json.dumps(args)[:200]}) -> {str(result).strip().splitlines()[0][:120] if str(result).strip() else ''}")
            messages.append({"role": "tool", "tool_call_id": tc["id"], "content": str(result)})

    print("stopped: reached max_turns")
    return None, messages

In [11]:
task = (
    "Create `stats.py` with functions `mean(values)`, `median(values)` and `mode(values)` for lists of numbers. "
    "Each must raise ValueError on empty input; `mode` should return the smallest value among ties. "
    "Then write `test_stats.py` using unittest covering normal cases, ties, a single element and empty input. "
    "Run the tests with `-m unittest -v`, fix anything that fails, and summarize what you built."
)

summary, transcript = run_agent(task)

print("\nfiles in workspace:\n" + list_files())
if (WORKSPACE / "stats.py").exists():
    print("\n--- stats.py ---\n" + read_file("stats.py"))
else:
    print("\nstats.py was not created — inspect `transcript` to see where the agent stopped")


======================== turn 1 ========================
[reasoning] Let me break down the task: 1. Create `stats.py` with three functions: `mean`, `median`, `mode` 2. Each function takes a list of numbers and raises ValueError on empty input 3. `mode` should return smallest value among ties 4. Create `test_stats.py` with unittest covering normal cases, ties, single …
[answer] I'll create the implementation and tests, then run them.
[tool] write_file({"path": "stats.py", "content": "def mean(values):\n    if not values:\n        raise ValueError(\"Cannot compute mean of empty list\")\n    return sum(values) / len(values)\n\n\ndef median(values):\n) -> wrote 889 characters to stats.py
[tool] write_file({"path": "test_stats.py", "content": "import unittest\nfrom stats import mean, median, mode\n\n\nclass TestStats(unittest.TestCase):\n    def test_mean_normal(self):\n        self.assertAlmostEqual(me) -> wrote 1406 characters to test_stats.py

======================== turn 2 ===========

## Optional: call the endpoint with the OpenAI SDK

SageMaker real-time endpoints also expose an [OpenAI-compatible path](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-openai-compatible.html). Any OpenAI-compatible client — the OpenAI SDK, LangChain, Strands Agents, or a coding-agent CLI that accepts a custom base URL — can talk to this inference component by changing only its base URL:

```
https://runtime.sagemaker.<region>.amazonaws.com/endpoints/<endpoint_name>/inference-components/<ic_name>/openai/v1
```

Authentication uses a short-lived bearer token (up to 12 hours) minted locally from your AWS credentials by `generate_token` in the `sagemaker-core` package (part of the SageMaker Python SDK v3, already present in SageMaker Studio images); no SigV4 wrapper is needed. On a kernel that still has SageMaker Python SDK v2 (`sagemaker<3`), upgrading `sagemaker-core` to 2.x overwrites the v2 package layout — run this section in a fresh environment or upgrade to `sagemaker>=3` deliberately. The calling identity needs `sagemaker:InvokeEndpoint` on this endpoint ARN and `sagemaker:CallWithBearerToken` with `Resource: "*"` (the latter does not support resource-level restrictions). Streaming works the same way, and the `reasoning` field comes through as an extra attribute on each delta.

In [12]:
%pip install --upgrade --quiet --no-warn-conflicts openai sagemaker-core

Note: you may need to restart the kernel to use updated packages.


In [13]:
import logging
from datetime import timedelta
from openai import OpenAI
from sagemaker.core.token_generator import generate_token

for name in ("httpx", "httpx2"):  # the HTTP clients log every request at INFO; keep the cell output readable
    logging.getLogger(name).setLevel(logging.WARNING)

base_url = (
    f"https://runtime.sagemaker.{region}.amazonaws.com"
    f"/endpoints/{endpoint_name}/inference-components/{ic_name}/openai/v1"
)
client = OpenAI(
    base_url=base_url,
    api_key=generate_token(region=region, expiry=timedelta(hours=1)),
    timeout=900,
    max_retries=0,  # as with the boto3 client: a retried generation would silently double the latency and GPU work
)

stream = client.chat.completions.create(
    model="",  # SageMaker routes on the URL; the field is passed through to vLLM
    messages=[
        {
            "role": "user",
            "content": "Write a bash one-liner that lists the 10 largest files under the current directory, then explain it in two sentences.",
        }
    ],
    max_tokens=2048,
    temperature=1.0,
    top_p=0.95,
    stream=True,
)

phase = None
for chunk in stream:
    if not chunk.choices:
        continue
    delta = chunk.choices[0].delta
    reasoning = getattr(delta, "reasoning", None) or getattr(delta, "reasoning_content", None)
    if reasoning:
        if phase != "reasoning":
            print("\n[reasoning]", flush=True)
            phase = "reasoning"
        print(reasoning, end="", flush=True)
    if delta.content:
        if phase != "answer":
            print("\n\n[answer]", flush=True)
            phase = "answer"
        print(delta.content, end="", flush=True)
print()


[reasoning]
 We need answer user: bash one-liner lists 10 largest files under current directory, then explain in two sentences. We need produce a reliable one-liner. Options: find . -type f -printf '%s %p\n' | sort -nr | head -10. This is GNU find specific. Alternative du -ah . | sort -rh | head -10 but that includes directories. Could use find . -type f -exec ls -lh {} + | sort -k5 -hr | head -10 but sorting human readable with GNU sort. A robust one-liner: find . -type f -printf '%s %p\n' | sort -nr | head -n 10. To show sizes human-readable: find . -type f -printf '%s %p\n' | sort -nr | head -n 10 | awk '{printf "%.2f MB\t%s\n", $1/1024/1024, $2}' but that breaks filenames with spaces. Better keep bytes or use human readable output: find . -type f -exec ls -lSh {} + | head -n 10 maybe includes dirs? -type f. But with many files command line length. Use xargs. For portability: find . -type f | xargs ls -lSh | head. But filenames with spaces problematic. GNU find -print0 xargs -0 ls 

## Notes

**Multi-turn conversations.** Keep `reasoning` on every assistant message you send back, as the agent loop does; the chat template accepts either `reasoning` or the legacy `reasoning_content` key. There is no instant (non-thinking) mode for this model. Do not pass `chat_template_kwargs={"enable_thinking": False}` (or `"thinking": False`): the chat template ignores these keys and still opens `<think>`, but vLLM 0.28.0's `kimi_k2` reasoning parser honors them and stops splitting reasoning from content, so the raw thinking text and its closing `</think>` leak into `content` and `reasoning` comes back empty (the same applies to `reasoning_effort: "none"`, which vLLM turns into `enable_thinking=False` for the parser; other `reasoning_effort` values are harmless for this template).

**Vision input.** The checkpoint includes the MoonViT encoder. To send images, delete `SM_VLLM_LANGUAGE_MODEL_ONLY`, add `SM_VLLM_MM_ENCODER_TP_MODE=data` and a `SM_VLLM_LIMIT_MM_PER_PROMPT` limit, redeploy, and pass OpenAI-style `image_url` content parts.


## Cleanup

Delete the inference component first — the endpoint cannot be deleted while a component is attached. The `ml.g7e.48xlarge` bills until the endpoint is gone.

In [ ]:
_ = sm.delete_inference_component(InferenceComponentName=ic_name)

# wait for the IC to detach before deleting the endpoint
while True:
    try:
        sm.describe_inference_component(InferenceComponentName=ic_name)
        time.sleep(20)
    except sm.exceptions.ClientError as exc:
        if exc.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
            break  # the component is gone
        raise

_ = sm.delete_endpoint(EndpointName=endpoint_name)
_ = sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
_ = sm.delete_model(ModelName=model_name)

if "WORKSPACE" in globals():  # only defined if the agent section was run
    shutil.rmtree(WORKSPACE, ignore_errors=True)